# RadarML Lite datathon — Kaggle Track B notebook

This notebook trains a **from-scratch time-frequency CNN** on the seven required families: `barker`, `cw`, `fmcw`, `fsk`, `hyp`, `nlfm`, `quad`. It uses the published source-pulse 80/20 split. It can read compact `*_recorded.npy` arrays or expanded `*_single_channel.npy` arrays; compact data receive the organizer's deterministic requested-SNR augmentation on the fly. It never uses a filename, pulse ID, metadata, or requested SNR as a model feature.

**Kaggle setup:** Attach the expanded dataset at `/kaggle/input/datasets/hadimanes/radar-ml/expanded` and the manifests at `/kaggle/input/datasets/yacinesm/manifests`. The path settings below are already filled in. The 660 expanded arrays need their JSON sidecars, and the manifest directory needs `split_manifest.csv`. The FSK filenames may be `fsk#0`, `fsk0`, `fsk_0`, or `fsk-0`; all four FSK subtypes map to the single `fsk` class. Enable a GPU and internet is not needed. Run all cells through validation. The last cell is for the organizer's blind release **only after model freeze and test release**.

The defaults train a manageable subset of eight training pulses per recording at five SNRs; increase `TRAIN_PULSES_PER_FILE` to 80 and `TRAIN_SNRS` to all 13 for a larger run. Validation defaults to all 20 published validation pulses and all 13 SNRs, so its reported primary metric is comparable to the event metric. Training on a subset must be disclosed.


In [ ]:
import csv
import hashlib
import json
import os
import random
import re
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

CLASSES = ("barker", "cw", "fmcw", "fsk", "hyp", "nlfm", "quad")
ALL_SNRS = (30, 25, 20, 15, 10, 5, 0, -5, -10, -15, -20, -25, -30)
SAMPLE_RATE_HZ = 120_000_000
SEED = 42
TRACK = "B"

# Your attached Kaggle datasets. MANIFEST_PATH may be a directory or CSV file.
DATA_ROOT = "/kaggle/input/datasets/hadimanes/radar-ml/expanded"
MANIFEST_PATH = "/kaggle/input/datasets/yacinesm/manifests"
OUTPUT_DIR = Path("/kaggle/working/radarml_track_b" if Path("/kaggle/working").exists() else "radarml_track_b")

RUN_TRAINING = True
TRAIN_PULSES_PER_FILE = 8     # 1..80; published training split only
TRAIN_SNRS = (30, 15, 0, -15, -30)
VAL_PULSES_PER_FILE = 20      # 20 gives the full published validation split
VAL_SNRS = ALL_SNRS
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
NUM_WORKERS = 2

# Leave unset during development. After the freeze, set RUN_TRAINING=False
# and point this at the organizer-issued blind package's test folder.
BLIND_ROOT = None

assert set(TRAIN_SNRS).issubset(ALL_SNRS) and set(VAL_SNRS).issubset(ALL_SNRS)
assert 1 <= TRAIN_PULSES_PER_FILE <= 80 and 1 <= VAL_PULSES_PER_FILE <= 20
assert BATCH_SIZE > 0 and EPOCHS > 0
assert not (RUN_TRAINING and BLIND_ROOT is not None), "Freeze first; do not train with blind data attached."
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| PyTorch:", torch.__version__, "| NumPy:", np.__version__)


## 1. Locate files and enforce the published split

The manifest names retain `#` while the Kaggle upload has collapsed multiple FSK filenames to names like `fsk0... (19).npy`. Each JSON sidecar retains its original `source_file` and the SHA256 hash of its array. **The number in the uploaded filename is not a reliable pairing key.** The lookup hashes FSK NPY files once, matches them to JSON by checksum, then maps the JSON's original source name to the published manifest. This changes only file lookup; it never changes split assignments. If a matching array is missing or ambiguous, the notebook stops with an error. Filenames are used solely to determine training labels and storage locations.


In [ ]:
def discover_data_root(explicit=None):
    if explicit is not None:
        root = Path(explicit)
        if not root.is_dir():
            raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")
        if not any(root.rglob("*.npy")):
            raise FileNotFoundError(f"No RadarML arrays below DATA_ROOT: {root}")
        return root
    bases = [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]
    for base in bases:
        if not base.exists():
            continue
        for pattern in ("*_single_channel.npy", "*_recorded.npy"):
            first = next(base.rglob(pattern), None)
            if first is not None:
                return first.parent
    raise FileNotFoundError("No RadarML arrays found. Attach the Kaggle dataset or set DATA_ROOT.")


def find_manifest(root, explicit=None):
    if explicit is not None:
        path = Path(explicit)
        if path.is_dir():
            matches = list(path.rglob("split_manifest.csv"))
            if len(matches) != 1:
                raise FileNotFoundError(f"Expected exactly one split_manifest.csv under {path}; found {len(matches)}")
            path = matches[0]
    else:
        candidates = [root / "split_manifest.csv", root.parent / "split_manifest.csv"]
        path = next((p for p in candidates if p.is_file()), None)
        if path is None and Path("/kaggle/input").exists():
            path = next(Path("/kaggle/input").rglob("split_manifest.csv"), None)
    if path is None or not path.is_file():
        raise FileNotFoundError("Published split_manifest.csv is required; add it to the Kaggle dataset or set MANIFEST_PATH.")
    return path


def family(filename):
    parts = Path(filename).stem.lower().split("_")
    if len(parts) < 3:
        raise ValueError(f"Unrecognized waveform filename: {filename}")
    token = parts[1]
    if token.startswith("barker"):
        return "barker"
    if re.fullmatch(r"fsk(?:[#-]?[0-3])?", token):
        return "fsk"
    if token == "fsk" and len(parts) > 2 and parts[2] in {"0", "1", "2", "3"}:
        return "fsk"
    if token in CLASSES:
        return token
    raise ValueError(f"Unrecognized modulation in {filename}: {token}")


def canonical_filename(filename):
    # Kaggle can append " (19)" before an extension. Preserve frequency signs.
    filename = re.sub(r" \(\d+\)(?=\.(?:npy|json)$)", "", filename)
    return re.sub(r"(?<=_)fsk[#_-]?([0-3])(?=_)", r"fsk\1", filename, flags=re.IGNORECASE)


def expanded_name_from_source(source_file):
    """Recreate the organizer's expanded name from JSON's original source_file."""
    source = Path(source_file).name
    if source.endswith("_recorded.npy"):
        return source.replace("_recorded.npy", "_single_channel.npy")
    if source.endswith("_single_channel.npy"):
        return source
    parts = Path(source).stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected source_file in JSON sidecar: {source}")
    try:
        parts[-1] = "-".join(f'{240 - int(piece.removesuffix("Mhz"))}Mhz'
                             for piece in parts[-1].split("-"))
    except ValueError as exc:
        raise ValueError(f"Unexpected source frequency in JSON sidecar: {source}") from exc
    return "_".join(parts) + "_single_channel.npy"


def build_file_lookup(root, kind):
    suffix = "_single_channel.npy" if kind == "expanded" else "_recorded.npy"
    files = [p for p in root.rglob("*.npy") if canonical_filename(p.name).endswith(suffix)]
    if not files:
        raise FileNotFoundError(f"No {suffix} files in {root}")
    file_set = set(files)
    arrays, metadata = defaultdict(set), defaultdict(list)
    for path in files:
        arrays[canonical_filename(path.name)].add(path)
    for sidecar in root.rglob("*.json"):
        meta = json.loads(sidecar.read_text(encoding="utf-8"))
        if not isinstance(meta, dict) or not meta.get("source_file"):
            continue
        alias = expanded_name_from_source(meta["source_file"])
        if kind == "compact":
            alias = alias.replace("_single_channel.npy", "_recorded.npy")
        key = canonical_filename(alias)
        metadata[key].append((sidecar, meta))
        paired_array = sidecar.with_suffix(".npy")
        if paired_array in file_set:
            arrays[key].add(paired_array)
    hash_to_paths, path_to_hash = defaultdict(list), {}
    if kind == "expanded":
        fsk_paths = [p for p in files if family(canonical_filename(p.name)) == "fsk"]
        print(f"Hashing {len(fsk_paths)} uploaded FSK arrays to recover their original identities...")
        for count, path in enumerate(fsk_paths, 1):
            digest = lookup_sha256(path)
            hash_to_paths[digest].append(path)
            path_to_hash[path] = digest
            if count % 25 == 0 or count == len(fsk_paths):
                print(f"Hashed {count}/{len(fsk_paths)} FSK arrays")
        for key, entries in metadata.items():
            if "_fsk" in key.lower():
                for _, meta in entries:
                    for path in hash_to_paths.get(meta.get("output_sha256"), []):
                        arrays[key].add(path)
    return {"arrays": arrays, "metadata": metadata,
            "hash_to_paths": hash_to_paths, "path_to_hash": path_to_hash}


def lookup_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resolve_file(manifest_name, lookup, kind):
    desired = (manifest_name.replace("_recorded.npy", "_single_channel.npy")
               if kind == "expanded" else manifest_name)
    key = canonical_filename(desired)
    if kind == "expanded" and family(manifest_name) == "fsk":
        expected = {meta.get("output_sha256") for _, meta in lookup["metadata"].get(key, [])}
        digests_found = {digest for digest in expected if digest in lookup["hash_to_paths"]}
        if len(digests_found) == 1:
            digest = next(iter(digests_found))
            return sorted(lookup["hash_to_paths"][digest])[0]
        raise ValueError(f"FSK array for {manifest_name} cannot be matched by SHA256. "
                         f"Matching hashes found: {len(digests_found)}. "
                         "The Kaggle dataset may be missing an NPY file or its JSON may be stale.")
    matches = list(lookup["arrays"].get(key, []))
    paired = {sidecar.with_suffix(".npy") for sidecar, _ in lookup["metadata"].get(key, [])}
    if paired:
        matches = [path for path in matches if path in paired]
    if matches:
        shallowest = min(len(path.parts) for path in matches)
        matches = [path for path in matches if len(path.parts) == shallowest]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        expected_hashes = {meta.get("output_sha256") for _, meta in lookup["metadata"].get(key, [])}
        matching = [(path, digest) for path in matches
                    if (digest := lookup_sha256(path)) in expected_hashes]
        if matching and len({digest for _, digest in matching}) == 1:
            return sorted(path for path, _ in matching)[0]
    examples = [p.name for paths in lookup["arrays"].values() for p in paths
                if "fsk" in p.name.lower()][:5]
    raise ValueError(f"Could not uniquely match {manifest_name}: {len(matches)} candidate arrays. "
                     f"Example Kaggle FSK names: {examples}. Check array uploads and JSON checksums.")


def resolve_metadata(manifest_name, path, lookup, kind):
    desired = (manifest_name.replace("_recorded.npy", "_single_channel.npy")
               if kind == "expanded" else manifest_name)
    candidates = lookup["metadata"].get(canonical_filename(desired), [])
    if kind == "expanded" and family(manifest_name) == "fsk":
        digest = lookup["path_to_hash"].get(path)
        matched = [(sidecar, meta) for sidecar, meta in candidates
                   if meta.get("output_sha256") == digest]
        if matched:
            return sorted(matched, key=lambda pair: str(pair[0]))[0]
        raise ValueError(f"FSK metadata checksum missing for {manifest_name}")
    exact = [(sidecar, meta) for sidecar, meta in candidates if sidecar == path.with_suffix(".json")]
    if len(exact) == 1:
        return exact[0]
    if len(candidates) == 1:
        return candidates[0]
    if candidates:
        digest = lookup_sha256(path)
        matched = [(sidecar, meta) for sidecar, meta in candidates
                   if meta.get("output_sha256") == digest]
        if matched:
            return sorted(matched, key=lambda pair: str(pair[0]))[0]
    raise ValueError(f"Could not pair {path.name} with a JSON sidecar for {manifest_name}")


def load_manifest(path):
    with path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["file", "pulse_index", "split"]:
            raise ValueError("Expected manifest columns: file,pulse_index,split")
        rows = list(reader)
    if not rows:
        raise ValueError("Empty manifest")
    identities = set()
    per_file = Counter()
    for row in rows:
        if row["split"] not in {"train", "validation"}:
            raise ValueError(f"Unexpected split {row['split']}")
        pulse = int(row["pulse_index"])
        if not 0 <= pulse < 100:
            raise ValueError(f"Pulse index outside 0..99: {pulse}")
        identity = (row["file"], pulse)
        if identity in identities:
            raise ValueError(f"Duplicate source pulse: {identity}")
        identities.add(identity)
        per_file[(row["file"], row["split"])] += 1
        row["pulse_index"] = pulse
    for name in {r["file"] for r in rows}:
        if per_file[(name, "train")] != 80 or per_file[(name, "validation")] != 20:
            raise ValueError(f"Manifest is not 80/20 for {name}")
    if len(rows) != 66_000 or len(per_file) != 1_320:
        raise ValueError("Expected the complete published 660-recording, 66,000-pulse manifest")
    print(f"Manifest: {len(rows):,} source pulses across {len(per_file)//2} recordings")
    return rows


if RUN_TRAINING:
    data_root = discover_data_root(DATA_ROOT)
    manifest_path = find_manifest(data_root, MANIFEST_PATH)
    manifest_rows = load_manifest(manifest_path)
    kind = ("expanded" if any(canonical_filename(p.name).endswith("_single_channel.npy")
                              for p in data_root.rglob("*.npy")) else "compact")
    lookup = build_file_lookup(data_root, kind)
    print("Data:", data_root, "| Layout:", kind, "| Split:", manifest_path)


In [ ]:
def make_file_info(manifest_rows, lookup, kind):
    infos = {}
    verified_fsk = 0
    for name in sorted({r["file"] for r in manifest_rows}):
        path = resolve_file(name, lookup, kind)
        sidecar, meta = resolve_metadata(name, path, lookup, kind)
        if kind == "expanded":
            snrs = tuple(int(s) for s in meta["snr_db"])
            if len(snrs) != len(set(snrs)):
                raise ValueError(f"Duplicate SNRs in {sidecar}")
        else:
            snrs = ()
            if not meta.get("source_file"):
                raise ValueError(f"Compact sidecar lacks source_file: {sidecar}")
        if int(meta["sample_rate_hz"]) != SAMPLE_RATE_HZ or meta.get("dtype") != "int16":
            raise ValueError(f"Unexpected acquisition settings: {sidecar}")
        if kind == "expanded" and family(name) == "fsk":
            expected_hash = meta.get("output_sha256")
            if not expected_hash or lookup["path_to_hash"].get(path) != expected_hash:
                raise ValueError(f"FSK NPY/JSON checksum mismatch: {path.name} / {sidecar.name}")
            verified_fsk += 1
        infos[name] = {"path": path, "label": family(name), "snrs": snrs,
                       "source_file": Path(meta.get("source_file", name)).name}
    if verified_fsk:
        print(f"Verified all {verified_fsk} FSK array/JSON pairs")
    return infos


def build_examples(rows, infos, split, pulse_limit, snrs):
    grouped = defaultdict(list)
    for row in rows:
        if row["split"] == split:
            grouped[row["file"]].append(row["pulse_index"])
    examples = []
    for name in sorted(grouped):
        info = infos[name]
        if info["snrs"] and not set(snrs).issubset(info["snrs"]):
            raise ValueError(f"Requested SNR missing from expanded sidecar: {info['path']}")
        # First N published pulse indices within each recording; deterministic.
        for pulse in sorted(grouped[name])[:pulse_limit]:
            for snr in snrs:
                examples.append((name, pulse, int(snr), CLASSES.index(info["label"])))
    return examples


if RUN_TRAINING:
    file_info = make_file_info(manifest_rows, lookup, kind)
    train_examples = build_examples(manifest_rows, file_info, "train", TRAIN_PULSES_PER_FILE, TRAIN_SNRS)
    val_examples = build_examples(manifest_rows, file_info, "validation", VAL_PULSES_PER_FILE, VAL_SNRS)
    print("Training examples:", f"{len(train_examples):,}", "| Validation examples:", f"{len(val_examples):,}")
    print("Training class counts:", {c: sum(e[3] == i for e in train_examples) for i, c in enumerate(CLASSES)})
    assert {e[3] for e in train_examples} == set(range(7))
    assert {e[3] for e in val_examples} == set(range(7))


## 2. Stream examples from disk

Compact arrays have shape `(1, 100, 33600)` and expanded arrays `(1, SNR, 100, 33600)`. The last axis interleaves I and Q. The compact path reproduces the organizer's noise calculation and deterministic seed convention; it computes one example at a time and clips to int16. The model receives only the two signal channels.


In [ ]:
def add_requested_noise(row, snr_db, source_file, pulse_index, seed=SEED):
        """Organizer's compact-to-expanded transformation for one interleaved row."""
        key = f"{seed}_{source_file}_{snr_db}_{pulse_index}"
        noise_seed = int(hashlib.md5(key.encode()).hexdigest()[:8], 16)
        rng = np.random.default_rng(noise_seed)
        iq = row[0::2].astype(np.float64) + 1j * row[1::2].astype(np.float64)
        noise_power = np.mean(np.abs(iq[:int(20e-6 * SAMPLE_RATE_HZ)]) ** 2)
        signal_power = np.mean(np.abs(iq[int(21e-6 * SAMPLE_RATE_HZ):int(119e-6 * SAMPLE_RATE_HZ)]) ** 2) - noise_power
        if not np.isfinite(signal_power) or signal_power <= 0:
            raise ValueError(f"Non-positive estimated signal power: {source_file}, pulse {pulse_index}")
        added_power = signal_power / (10.0 ** (snr_db / 10.0)) - noise_power
        sigma = np.sqrt(max(0.0, added_power) / 2.0)
        noisy = iq + rng.normal(0, sigma, iq.size) + 1j * rng.normal(0, sigma, iq.size)
        out = np.empty(row.size, dtype=np.float64)
        out[0::2], out[1::2] = noisy.real, noisy.imag
        return np.clip(out, -32768, 32767).astype(np.int16)


def interleaved_to_channels(row):
    if row.shape != (33600,) or row.dtype != np.int16:
        raise ValueError(f"Expected interleaved int16 row of length 33600, got {row.shape} {row.dtype}")
    return np.stack((row[0::2], row[1::2])).astype(np.float32) / 32768.0


class DevelopmentDataset(Dataset):
    def __init__(self, examples, file_info, kind):
        self.examples, self.file_info, self.kind = examples, file_info, kind

    @staticmethod
    @lru_cache(maxsize=16)
    def _array(path):
        return np.load(path, mmap_mode="r", allow_pickle=False)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        name, pulse, snr, label = self.examples[index]
        info = self.file_info[name]
        array = self._array(str(info["path"]))
        if self.kind == "expanded":
            if array.ndim != 4 or array.shape[0] != 1 or array.shape[-1] != 33600:
                raise ValueError(f"Bad expanded array shape: {info['path']}: {array.shape}")
            row = array[0, info["snrs"].index(snr), pulse, :]
        else:
            if array.ndim != 3 or array.shape[0] != 1 or array.shape[-1] != 33600:
                raise ValueError(f"Bad compact array shape: {info['path']}: {array.shape}")
            row = add_requested_noise(array[0, pulse, :], snr, info["source_file"], pulse)
        return torch.from_numpy(interleaved_to_channels(row)), label, snr


if RUN_TRAINING:
    training_data = DevelopmentDataset(train_examples, file_info, kind)
    validation_data = DevelopmentDataset(val_examples, file_info, kind)
    x0, y0, s0 = training_data[0]
    assert x0.shape == (2, 16800) and x0.dtype == torch.float32
    print("Example:", tuple(x0.shape), CLASSES[y0], "requested SNR", s0, "dB")


## 3. Track B model and event metric

The network computes a two-sided STFT from I/Q inside `forward`, so the exact same preprocessing runs in training and inference. It uses no pretrained weights. The score averages seven class recalls at each requested SNR, then averages the 13 SNR scores. Rows of each confusion matrix are true labels; columns are predictions.


In [ ]:
class SpectrogramCNN(nn.Module):
        def __init__(self, num_classes=7, n_fft=256, hop_length=128):
            super().__init__()
            self.n_fft, self.hop_length = n_fft, hop_length
            self.register_buffer("window", torch.hann_window(n_fft), persistent=False)
            layers = []
            channels = (1, 16, 32, 64, 128)
            for in_ch, out_ch in zip(channels[:-1], channels[1:]):
                layers += [nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1),
                           nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            self.features = nn.Sequential(*layers)
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
                                      nn.Linear(128 * 4 * 4, 128), nn.ReLU(inplace=True),
                                      nn.Linear(128, num_classes))

        def forward(self, x):
            # Fixed per-pulse RMS normalization is permitted by the event rules.
            rms = torch.sqrt(torch.mean(x.square(), dim=(1, 2), keepdim=True)).clamp_min(1e-6)
            x = x / rms
            iq = torch.complex(x[:, 0], x[:, 1])
            stft = torch.stft(iq, n_fft=self.n_fft, hop_length=self.hop_length,
                              win_length=self.n_fft, window=self.window, center=False,
                              onesided=False, return_complex=True)
            spectrum = torch.log1p(torch.abs(torch.fft.fftshift(stft, dim=1))).unsqueeze(1)
            mean = spectrum.mean(dim=(2, 3), keepdim=True)
            std = spectrum.std(dim=(2, 3), keepdim=True).clamp_min(1e-6)
            return self.head(self.features((spectrum - mean) / std))


def score_predictions(y_true, y_pred, snrs, expected_snrs=ALL_SNRS):
    matrices = {int(s): np.zeros((7, 7), dtype=np.int64) for s in expected_snrs}
    for truth, prediction, snr in zip(y_true, y_pred, snrs):
        matrices[int(snr)][int(truth), int(prediction)] += 1
    per_snr = []
    for snr in expected_snrs:
        matrix = matrices[int(snr)]
        totals = matrix.sum(axis=1)
        if np.any(totals == 0):
            raise ValueError(f"Validation SNR {snr} lacks one or more classes")
        per_snr.append({"requested_snr_db": int(snr),
                        "balanced_accuracy": float(np.mean(matrix.diagonal() / totals)),
                        "accuracy": float(matrix.trace() / matrix.sum()),
                        "confusion_matrix": matrix.tolist()})
    return {"classes": list(CLASSES), "samples": len(y_true),
            "mean_balanced_accuracy": float(np.mean([r["balanced_accuracy"] for r in per_snr])),
            "per_snr": per_snr}


@torch.no_grad()
def evaluate(model, loader, expected_snrs):
    model.eval()
    truths, predictions, snrs = [], [], []
    for signals, labels, batch_snrs in loader:
        logits = model(signals.to(DEVICE))
        truths.extend(labels.tolist())
        predictions.extend(logits.argmax(1).cpu().tolist())
        snrs.extend(batch_snrs.tolist())
    return score_predictions(truths, predictions, snrs, expected_snrs)


In [ ]:
def train_model(training_data, validation_data):
        labels = np.array([example[3] for example in training_data.examples])
        counts = np.bincount(labels, minlength=len(CLASSES))
        if np.any(counts == 0):
            raise ValueError("Training subset must contain all seven classes")
        weights = (1.0 / counts[labels]).astype(np.float64)
        sampler = WeightedRandomSampler(torch.as_tensor(weights), len(weights), replacement=True,
                                        generator=torch.Generator().manual_seed(SEED))
        train_loader = DataLoader(training_data, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
        val_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
        model = SpectrogramCNN().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.CrossEntropyLoss()
        best = -1.0
        started = time.perf_counter()
        for epoch in range(1, EPOCHS + 1):
            model.train()
            loss_total, seen = 0.0, 0
            for signals, labels_batch, _ in train_loader:
                signals, labels_batch = signals.to(DEVICE), labels_batch.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(signals), labels_batch)
                loss.backward()
                optimizer.step()
                loss_total += loss.item() * len(labels_batch)
                seen += len(labels_batch)
            report = evaluate(model, val_loader, VAL_SNRS)
            score = report["mean_balanced_accuracy"]
            print(f"Epoch {epoch}/{EPOCHS}: loss={loss_total/seen:.4f}, validation mean balanced accuracy={score:.4f}")
            if score > best:
                best = score
                torch.save({"state_dict": model.state_dict(), "classes": CLASSES,
                            "track": TRACK, "n_fft": model.n_fft, "hop_length": model.hop_length},
                           OUTPUT_DIR / "model.pt")
                report["selected_epoch"] = epoch
                (OUTPUT_DIR / "validation_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
        return best, time.perf_counter() - started


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


if RUN_TRAINING:
    best_score, training_seconds = train_model(training_data, validation_data)
    configuration = {
        "track": TRACK, "classes": CLASSES, "data_layout": kind,
        "dataset_root": str(data_root), "manifest_sha256": sha256(manifest_path),
        "seed": SEED, "selection_rule": "first N published pulse indices per recording and split",
        "train_pulses_per_file": TRAIN_PULSES_PER_FILE, "train_snrs_db": TRAIN_SNRS,
        "validation_pulses_per_file": VAL_PULSES_PER_FILE, "validation_snrs_db": VAL_SNRS,
        "batch_size": BATCH_SIZE, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
        "fsk_identity_check": "SHA256 of uploaded NPY matched to JSON output_sha256",
        "sample_rate_hz": SAMPLE_RATE_HZ, "n_fft": 256, "hop_length": 128,
        "best_validation_mean_balanced_accuracy": best_score,
        "training_seconds": training_seconds, "device": str(DEVICE),
        "python": sys.version, "torch": torch.__version__, "numpy": np.__version__,
        "model_size_bytes": (OUTPUT_DIR / "model.pt").stat().st_size,
    }
    config_path = OUTPUT_DIR / "config.json"
    config_path.write_text(json.dumps(configuration, indent=2), encoding="utf-8")
    hashes = {name: sha256(OUTPUT_DIR / name) for name in ("model.pt", "config.json")}
    (OUTPUT_DIR / "sha256.json").write_text(json.dumps(hashes, indent=2), encoding="utf-8")
    print("Best validation score:", round(best_score, 4))
    print("Training time (s):", round(training_seconds, 1))
    print("Saved files:", OUTPUT_DIR, "| SHA256:", hashes)


## 4. Review validation evidence and freeze

Review the SNR curve and confusion matrices, then retain `model.pt`, `config.json`, `validation_report.json`, `sha256.json`, and this notebook. The score is the **full published validation metric** only when `VAL_PULSES_PER_FILE=20` and `VAL_SNRS=ALL_SNRS`. Freeze one selected pipeline by Day 4, 17:00 Europe/London. Report your hardware, training time, selected subset, attribution (including AI assistance), and validation results in the method summary.


In [ ]:
if RUN_TRAINING:
    report = json.loads((OUTPUT_DIR / "validation_report.json").read_text(encoding="utf-8"))
    print("Mean balanced accuracy:", f"{report['mean_balanced_accuracy']:.4f}")
    print("Requested SNR (dB) | Balanced accuracy")
    for item in report["per_snr"]:
        print(f"{item['requested_snr_db']:+3d} | {item['balanced_accuracy']:.4f}")
    x = [r["requested_snr_db"] for r in report["per_snr"]]
    y = [r["balanced_accuracy"] for r in report["per_snr"]]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, y, marker="o")
    ax.set(xlabel="Requested SNR (dB)", ylabel="Seven-class balanced accuracy",
           ylim=(0, 1), title="Published validation split")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "validation_snr_curve.png", dpi=150)
    plt.show()

    shown_snrs = [s for s in (30, 0, -30) if s in VAL_SNRS]
    fig, axes = plt.subplots(1, len(shown_snrs), figsize=(5 * len(shown_snrs), 4))
    for ax, snr in zip(np.atleast_1d(axes), shown_snrs):
        matrix = np.array(next(r["confusion_matrix"] for r in report["per_snr"]
                               if r["requested_snr_db"] == snr), dtype=float)
        matrix /= matrix.sum(axis=1, keepdims=True)
        ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
        ax.set(title=f"{snr:+d} dB", xlabel="Predicted", ylabel="True")
        ax.set_xticks(range(7), CLASSES, rotation=45, ha="right")
        ax.set_yticks(range(7), CLASSES)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "validation_confusions.png", dpi=150)
    plt.show()


## 5. Blind inference after organizer release

After freeze and after the official blind package is released, set `RUN_TRAINING=False`, set `BLIND_ROOT` to the folder containing `index.csv` and `submission_template.csv`, and run the notebook with the **same frozen `model.pt`** in `OUTPUT_DIR`. Keep `sha256.json` there too; the inference function verifies its frozen model hash. This path makes no training or validation-data access. It writes exactly `sample_id,predicted_label`, checks all IDs against the template, and measures inference time including file loading and STFT preprocessing. Do not use blind signals to select a model or change preprocessing.

Reproducible inference command after editing those two settings and attaching the frozen artifacts: `jupyter nbconvert --execute --to notebook radarml_lite_kaggle_track_b.ipynb --output blind_inference_run.ipynb`. The Kaggle notebook **Run All** action executes the same cells.


In [ ]:
class BlindDataset(Dataset):
        def __init__(self, root, rows):
            self.root, self.rows = Path(root), rows

        @staticmethod
        @lru_cache(maxsize=16)
        def _array(path):
            return np.load(path, mmap_mode="r", allow_pickle=False)

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, index):
            info = self.rows[index]
            array = self._array(str(self.root / info["file"]))
            row = array[int(info["row"])]
            return torch.from_numpy(interleaved_to_channels(row))


def load_frozen_model(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if tuple(checkpoint["classes"]) != CLASSES or checkpoint["track"] != "B":
        raise ValueError("Frozen checkpoint has unexpected classes or track")
    model = SpectrogramCNN(n_fft=int(checkpoint["n_fft"]),
                           hop_length=int(checkpoint["hop_length"]))
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(DEVICE).eval()


def predict_blind(blind_root, checkpoint_path, output_path):
    root = Path(blind_root)
    frozen_hashes = Path(checkpoint_path).with_name("sha256.json")
    if frozen_hashes.is_file():
        expected_hash = json.loads(frozen_hashes.read_text(encoding="utf-8"))["model.pt"]
        if sha256(checkpoint_path) != expected_hash:
            raise ValueError("Frozen model hash differs from sha256.json")
    if not (root / "index.csv").is_file() and (root / "test" / "index.csv").is_file():
        root = root / "test"
    with (root / "index.csv").open(newline="", encoding="utf-8-sig") as stream:
        index_rows = list(csv.DictReader(stream))
    template = root / "submission_template.csv"
    if not template.is_file():
        template = root.parent / "submission_template.csv"
    with template.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["sample_id", "predicted_label"]:
            raise ValueError("Template must have sample_id,predicted_label columns")
        template_ids = [row["sample_id"] for row in reader]
    index_ids = [row["sample_id"] for row in index_rows]
    if (not index_ids or len(set(index_ids)) != len(index_ids) or
            len(set(template_ids)) != len(template_ids) or set(index_ids) != set(template_ids)):
        raise ValueError("Blind index and submission template IDs disagree or contain duplicates")
    model = load_frozen_model(checkpoint_path)
    loader = DataLoader(BlindDataset(root, index_rows), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=DEVICE.type == "cuda")
    started = time.perf_counter()
    predicted = []
    with torch.inference_mode():
        for signals in loader:
            predicted.extend(model(signals.to(DEVICE)).argmax(1).cpu().tolist())
    if len(predicted) != len(index_ids):
        raise RuntimeError("Prediction count differs from blind index")
    by_id = dict(zip(index_ids, (CLASSES[i] for i in predicted)))
    with Path(output_path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(("sample_id", "predicted_label"))
        writer.writerows((sample_id, by_id[sample_id]) for sample_id in template_ids)
    elapsed = time.perf_counter() - started
    return {"samples": len(index_ids), "inference_seconds": elapsed,
            "seconds_per_pulse": elapsed / len(index_ids),
            "checkpoint_sha256": sha256(checkpoint_path), "submission_sha256": sha256(output_path)}


if BLIND_ROOT is not None:
    result = predict_blind(BLIND_ROOT, OUTPUT_DIR / "model.pt", OUTPUT_DIR / "predictions.csv")
    (OUTPUT_DIR / "inference_report.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("Submission:", OUTPUT_DIR / "predictions.csv")
    print(json.dumps(result, indent=2))
else:
    print("Blind inference skipped; set BLIND_ROOT only after the model freeze and official release.")
